# Camada Gold com PySpark, cloud e FinOps

Depende das camadas geradas nos notebooks 01 e 02.

Cinco etapas: sessão Spark configurada para custo e performance, as 8 tabelas da Gold, otimização com
tempos medidos, FinOps com medição de compressão e bytes lidos, e geração dos jobs Glue, DDLs do
Athena e Lambda de streaming.

Sobre usar PySpark quando pandas daria conta: o volume real da fonte é maior do que parece, já que os
microdados de aluno passam de 3,8 milhões de linhas, e Spark é o mesmo motor do AWS Glue, o que faz o
código desta camada virar job sem reescrita. O contraponto é o overhead de inicialização, que não se
paga em dado pequeno. Por isso a divisão: pandas nas dimensões, Spark nos fatos e na Gold.

In [ ]:
# ============================================================
# ETAPA 1.1 — MOTOR DE EXECUÇÃO: SPARK (padrão) OU DUCKDB (plano B)
# ============================================================
# A Gold é escrita em SQL, então roda em qualquer motor: Spark quando está
# disponível, DuckDB quando não há Java instalado. O resultado é o mesmo.

import sys

IN_COLAB = "google.colab" in sys.modules

try:
    import pyspark
    MOTOR = "spark"
    print(f"Motor: SPARK {pyspark.__version__}")
except ImportError:
    if IN_COLAB:
        !pip install pyspark==3.5.1 --quiet
        import pyspark
        MOTOR = "spark"
        print(f"Motor: SPARK {pyspark.__version__} (instalado agora)")
    else:
        !pip install duckdb --quiet
        import duckdb
        MOTOR = "duckdb"
        print(f"Motor: DUCKDB {duckdb.__version__} (PySpark indisponível neste ambiente)")

!pip install pyarrow tabulate --quiet

In [ ]:
# ============================================================
# ETAPA 1.2 — CONTEXTO DO PROJETO
# ============================================================
import json
import logging
import os
import shutil
import subprocess
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")

REPO_NAME = "tech-challenge-fase2"
REPO_URL  = "https://github.com/eriicpa/tech-challenge-fase2.git"


def localizar_raiz_projeto():
    for candidato in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidato / "data" / "landing").is_dir():
            return candidato
    if IN_COLAB:
        destino = Path("/content") / REPO_NAME
        if not destino.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(destino)], check=True)
        return destino
    raise RuntimeError("Raiz do projeto não encontrada — abra o notebook dentro do repositório.")


PROJECT_ROOT = localizar_raiz_projeto()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

LANDING = Path("data/landing")
LAKE    = Path(os.environ.get("TC2_LAKE", "data/lake"))
BRONZE  = LAKE / "bronze"
SILVER  = LAKE / "silver"
GOLD    = LAKE / "gold"
QUALITY = LAKE / "quality"
MONITOR = LAKE / "monitoring"
GOLD.mkdir(parents=True, exist_ok=True)

AWS_ACCOUNT_ID = os.environ.get("AWS_ACCOUNT_ID", "000000000000")
AWS_REGION     = "us-east-1"
S3_PREFIX = {
    "bronze": f"s3://{AWS_ACCOUNT_ID}-tc2-data-sor/bronze",
    "silver": f"s3://{AWS_ACCOUNT_ID}-tc2-data-sot/silver",
    "gold":   f"s3://{AWS_ACCOUNT_ID}-tc2-data-spec/gold",
}
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)-8s | %(message)s",
                    datefmt="%Y-%m-%dT%H:%M:%SZ", force=True)
log = logging.getLogger("pipeline.gold")

if not (SILVER / "fato_alfabetizacao_municipio").exists():
    raise RuntimeError("Camada Silver não encontrada. Execute o Notebook 01 antes deste.")

print(f"Raiz do projeto: {PROJECT_ROOT}")
print(f"Silver: {SILVER.resolve()}")
print(f"Gold  : {GOLD.resolve()}  ->  {S3_PREFIX['gold']}")

In [ ]:
# ============================================================
# ETAPA 1.3 — SESSÃO DE PROCESSAMENTO: CADA CONFIG É UMA DECISÃO DE CUSTO
# ============================================================
spark = None
conexao_duckdb = None

if MOTOR == "spark":
    from pyspark.sql import SparkSession
    from pyspark.sql import functions as F
    from pyspark.sql.functions import broadcast

    spark = (
        SparkSession.builder
        .appName("tc2-alfabetizacao-gold")

        # AQE: o Spark refaz o plano durante a execução, com base no dado real.
        # Ajuda quando o join está mal dimensionado ou a partição desbalanceada.
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .config("spark.sql.adaptive.skewJoin.enabled", "true")

        # O padrão são 200 partições de shuffle, o que para este volume gera
        # arquivos pequenos demais.
        .config("spark.sql.shuffle.partitions", "8")

        # Sobrescreve só a partição afetada, em vez da tabela toda.
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")

        # Snappy comprime bem sem gastar muita CPU.
        .config("spark.sql.parquet.compression.codec", "snappy")

        # Empurra o filtro para o arquivo parquet, lendo menos bytes.
        .config("spark.sql.parquet.filterPushdown", "true")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("WARN")

    print(f"Spark {spark.version}")
    print(f"  shuffle partitions  : {spark.conf.get('spark.sql.shuffle.partitions')}")
    print(f"  AQE                 : {spark.conf.get('spark.sql.adaptive.enabled')}")
    print(f"  partitionOverwrite  : {spark.conf.get('spark.sql.sources.partitionOverwriteMode')}")
else:
    import duckdb
    conexao_duckdb = duckdb.connect()
    print("DuckDB pronto (execução single-node, mesmas consultas SQL).")

---
## Etapa 2 - Leitura da Silver

O Spark descobre as partições pela estrutura de diretórios (`ano=2023/`, `ano=2024/`). É essa
descoberta que habilita o partition pruning: um `WHERE ano = 2024` faz o motor nem abrir os arquivos
de 2023.

Cada tabela Silver é registrada como view SQL. A partir daí toda a lógica da Gold é SQL, o que
permite reaproveitar as mesmas consultas como views do Athena.

In [ ]:
# ============================================================
# ETAPA 2.1 — VIEWS DA CAMADA SILVER
# ============================================================
from src.utils.lakeio import ler_parquet

VIEWS_SILVER = {
    "silver_alfabetizacao_municipio": "fato_alfabetizacao_municipio",
    "silver_indicador_uf":            "fato_indicador_uf",
    "silver_dim_municipio":           "dim_municipio",
    "silver_dim_uf":                  "dim_uf",
    "silver_meta_municipio":          "dim_meta_municipio",
    "silver_meta_uf":                 "dim_meta_uf",
    "silver_distribuicao_nivel":      "fato_distribuicao_nivel",
    "silver_aluno":                   "fato_aluno",
    "silver_avaliacao_stream":        "fato_avaliacao_stream",   # produzida pelo Notebook 02
}

views_registradas = {}
for view, tabela in VIEWS_SILVER.items():
    caminho = SILVER / tabela
    if not caminho.exists():
        log.warning("Tabela Silver ausente (será ignorada): %s", tabela)
        continue

    if MOTOR == "spark":
        df = spark.read.parquet(str(caminho))
        df.createOrReplaceTempView(view)
        views_registradas[view] = df.count()
    else:
        conexao_duckdb.register(view, ler_parquet(caminho))
        views_registradas[view] = conexao_duckdb.execute(
            f"SELECT COUNT(*) FROM {view}").fetchone()[0]

    log.info("View registrada: %-32s | %8d linhas", view, views_registradas[view])


def executar_sql(sql):
    """Executa uma consulta no motor ativo e devolve o resultado (Spark DF ou pandas DF)."""
    return spark.sql(sql) if MOTOR == "spark" else conexao_duckdb.execute(sql).df()


print(f"\n{len(views_registradas)} views registradas no motor {MOTOR.upper()}")
display(pd.DataFrame(sorted(views_registradas.items()), columns=["view", "linhas"]))

---
## Etapa 3 - Camada Gold

A Gold é organizada por pergunta de negócio. Cada tabela existe porque um gestor, um dashboard ou um
modelo precisa dela pronta, sem ter que fazer join nem conhecer o dicionário de códigos do Inep.

São 8 tabelas, declaradas em `src/gold/consultas.py` com a consulta, o particionamento e a pergunta
que respondem. A listagem sai no resultado da próxima célula.

Duas regras seguidas aqui. Se a tabela traz `rede`, traz também `rede_nome`, porque quem consome a
Gold não deveria precisar de dicionário. E métrica derivada, como risco, situação da meta e ritmo
necessário até 2030, é calculada uma única vez neste ponto, para que dois relatórios da mesma
instituição não divirjam.

In [ ]:
%%writefile src/gold/consultas.py
"""Consultas que constroem a camada Gold.

Escrevemos em SQL, e não na DSL do Spark, por portabilidade (o mesmo texto roda
no Spark, no Athena e no DuckDB) e porque a regra de negócio fica legível para
quem entende de educação, não só para quem entende Python.

Cada entrada declara a consulta, as colunas de partição e a pergunta que responde.
"""

CONSULTAS_GOLD = {

    # ------------------------------------------------------------------
    "gold_indicador_municipio": {
        "descricao": "Indicador por município, ano e rede, com meta, risco e posição relativa.",
        "pergunta": "Como está cada município e quão longe ele está da própria meta?",
        "particoes": ["ano"],
        "sql": """
            SELECT
                ano,
                id_municipio,
                nome_municipio,
                sigla_uf,
                nome_uf,
                nome_regiao,
                nome_mesorregiao,
                rede,
                rede_nome,
                serie,
                taxa_alfabetizacao,
                media_portugues,
                percentual_participacao,
                meta_taxa_ano,
                meta_taxa_2030,
                gap_meta_ano,
                gap_meta_2030,
                taxa_uf,
                taxa_brasil_publica,
                dif_vs_uf,
                dif_vs_brasil,
                CASE
                    WHEN taxa_alfabetizacao < 40 THEN 'Critico'
                    WHEN taxa_alfabetizacao < 55 THEN 'Alto'
                    WHEN taxa_alfabetizacao < 70 THEN 'Medio'
                    ELSE 'Baixo'
                END AS risco_alfabetizacao,
                CASE
                    WHEN meta_taxa_ano IS NULL THEN 'Sem meta definida'
                    WHEN atingiu_meta_ano THEN 'Meta atingida'
                    ELSE 'Abaixo da meta'
                END AS situacao_meta,
                ROUND(gap_meta_2030 / NULLIF(2030 - ano, 0), 2) AS ritmo_anual_necessario_pp,
                RANK() OVER (
                    PARTITION BY ano, rede, sigla_uf ORDER BY taxa_alfabetizacao DESC
                ) AS posicao_na_uf,
                NTILE(4) OVER (
                    PARTITION BY ano, rede ORDER BY taxa_alfabetizacao
                ) AS quartil_nacional
            FROM silver_alfabetizacao_municipio
        """,
    },

    # ------------------------------------------------------------------
    "gold_meta_vs_realizado": {
        "descricao": "Consolidado meta × realizado por ano, rede e UF.",
        "pergunta": "Quais estados estão na rota da meta e quais concentram municípios críticos?",
        "particoes": ["ano"],
        "sql": """
            SELECT
                ano,
                rede,
                rede_nome,
                sigla_uf,
                nome_regiao,
                COUNT(DISTINCT id_municipio) AS municipios,
                ROUND(AVG(taxa_alfabetizacao), 2) AS taxa_media,
                ROUND(AVG(meta_taxa_ano), 2) AS meta_media,
                ROUND(AVG(gap_meta_ano), 2) AS gap_medio_pp,
                ROUND(AVG(gap_meta_2030), 2) AS gap_medio_2030_pp,
                ROUND(
                    100.0 * SUM(CASE WHEN atingiu_meta_ano THEN 1 ELSE 0 END)
                    / NULLIF(SUM(CASE WHEN meta_taxa_ano IS NOT NULL THEN 1 ELSE 0 END), 0), 1
                ) AS pct_municipios_na_meta,
                SUM(CASE WHEN taxa_alfabetizacao < 40 THEN 1 ELSE 0 END) AS municipios_criticos,
                ROUND(AVG(percentual_participacao), 1) AS participacao_media
            FROM silver_alfabetizacao_municipio
            GROUP BY ano, rede, rede_nome, sigla_uf, nome_regiao
        """,
    },

    # ------------------------------------------------------------------
    "gold_evolucao_temporal": {
        "descricao": "Variação do indicador entre o primeiro e o último ciclo disponível.",
        "pergunta": "Quem melhorou, quem piorou e em quantos anos cada município chega à meta?",
        "particoes": None,
        "sql": """
            WITH marcos AS (
                SELECT id_municipio, rede,
                       MIN(ano) AS ano_inicial,
                       MAX(ano) AS ano_final
                FROM silver_alfabetizacao_municipio
                GROUP BY id_municipio, rede
            ),
            consolidado AS (
                SELECT
                    s.id_municipio,
                    s.rede,
                    s.rede_nome,
                    m.ano_inicial,
                    m.ano_final,
                    MAX(CASE WHEN s.ano = m.ano_inicial THEN s.taxa_alfabetizacao END) AS taxa_inicial,
                    MAX(CASE WHEN s.ano = m.ano_final   THEN s.taxa_alfabetizacao END) AS taxa_final,
                    MAX(CASE WHEN s.ano = m.ano_final   THEN s.meta_taxa_2030 END)     AS meta_2030,
                    MAX(CASE WHEN s.ano = m.ano_final   THEN s.nome_municipio END)     AS nome_municipio,
                    MAX(CASE WHEN s.ano = m.ano_final   THEN s.sigla_uf END)           AS sigla_uf,
                    MAX(CASE WHEN s.ano = m.ano_final   THEN s.nome_regiao END)        AS nome_regiao
                FROM silver_alfabetizacao_municipio s
                JOIN marcos m
                  ON s.id_municipio = m.id_municipio AND s.rede = m.rede
                GROUP BY s.id_municipio, s.rede, s.rede_nome, m.ano_inicial, m.ano_final
            )
            SELECT
                id_municipio,
                nome_municipio,
                sigla_uf,
                nome_regiao,
                rede,
                rede_nome,
                ano_inicial,
                ano_final,
                taxa_inicial,
                taxa_final,
                meta_2030,
                ROUND(taxa_final - taxa_inicial, 2) AS variacao_pp,
                ROUND((taxa_final - taxa_inicial) / NULLIF(ano_final - ano_inicial, 0), 2)
                    AS variacao_media_anual_pp,
                CASE
                    WHEN ano_inicial = ano_final THEN 'Serie unica'
                    WHEN taxa_final - taxa_inicial > 1 THEN 'Melhora'
                    WHEN taxa_final - taxa_inicial < -1 THEN 'Piora'
                    ELSE 'Estavel'
                END AS tendencia,
                CASE
                    WHEN taxa_final >= meta_2030 THEN 0
                    WHEN taxa_final > taxa_inicial AND ano_final > ano_inicial
                        THEN ROUND(
                            (meta_2030 - taxa_final)
                            / ((taxa_final - taxa_inicial) / (ano_final - ano_inicial)), 1)
                    ELSE NULL
                END AS anos_estimados_para_meta
            FROM consolidado
        """,
    },

    # ------------------------------------------------------------------
    "gold_ranking_uf": {
        "descricao": "Indicador estadual com posição nacional e distância da meta da UF.",
        "pergunta": "Qual a fotografia por unidade da federação?",
        "particoes": ["ano"],
        "sql": """
            SELECT
                u.ano,
                u.sigla_uf,
                u.nome_uf,
                u.nome_regiao,
                u.rede,
                u.rede_nome,
                u.taxa_alfabetizacao AS taxa_uf,
                u.media_portugues,
                mu.meta_taxa AS meta_taxa_ano,
                ROUND(u.taxa_alfabetizacao - mu.meta_taxa, 2) AS gap_meta_ano,
                RANK() OVER (
                    PARTITION BY u.ano, u.rede ORDER BY u.taxa_alfabetizacao DESC
                ) AS posicao_nacional
            FROM silver_indicador_uf u
            LEFT JOIN silver_meta_uf mu
              ON mu.sigla_uf = u.sigla_uf
             AND mu.rede = u.rede
             AND mu.ano_meta = u.ano
        """,
    },

    # ------------------------------------------------------------------
    "gold_distribuicao_niveis": {
        "descricao": "Distribuição média dos alunos por nível de proficiência, por UF.",
        "pergunta": "A dificuldade está concentrada nos níveis mais baixos ou é difusa?",
        "particoes": ["ano"],
        "sql": """
            SELECT
                n.ano,
                m.sigla_uf,
                m.nome_regiao,
                n.rede,
                n.rede_nome,
                n.nivel,
                ROUND(AVG(n.proporcao_alunos), 2) AS proporcao_media,
                COUNT(DISTINCT n.id_municipio) AS municipios
            FROM silver_distribuicao_nivel n
            JOIN silver_dim_municipio m
              ON n.id_municipio = m.id_municipio
            GROUP BY n.ano, m.sigla_uf, m.nome_regiao, n.rede, n.rede_nome, n.nivel
        """,
    },

    # ------------------------------------------------------------------
    "gold_perfil_aluno_municipio": {
        "descricao": "Agregação ponderada dos microdados de aluno por município e rede.",
        "pergunta": "Qual a dispersão de proficiência e a participação dentro de cada município?",
        "particoes": ["ano"],
        "sql": """
            SELECT
                ano,
                id_municipio,
                rede,
                rede_nome,
                COUNT(*) AS alunos_cadastrados,
                SUM(CASE WHEN avaliado THEN 1 ELSE 0 END) AS alunos_avaliados,
                ROUND(100.0 * SUM(CASE WHEN avaliado THEN 1 ELSE 0 END) / COUNT(*), 2)
                    AS taxa_participacao,
                ROUND(SUM(CASE WHEN avaliado THEN proficiencia_portugues * peso_aluno END)
                      / NULLIF(SUM(CASE WHEN avaliado THEN peso_aluno END), 0), 2)
                    AS proficiencia_media,
                ROUND(STDDEV(CASE WHEN avaliado THEN proficiencia_portugues END), 2)
                    AS proficiencia_desvio,
                ROUND(MIN(proficiencia_portugues), 2) AS proficiencia_minima,
                ROUND(MAX(proficiencia_portugues), 2) AS proficiencia_maxima,
                ROUND(100.0 * SUM(CASE WHEN avaliado AND alfabetizado THEN peso_aluno END)
                      / NULLIF(SUM(CASE WHEN avaliado THEN peso_aluno END), 0), 2)
                    AS taxa_alfabetizacao_microdados
            FROM silver_aluno
            GROUP BY ano, id_municipio, rede, rede_nome
        """,
    },

    # ------------------------------------------------------------------
    "gold_features_ml": {
        "descricao": "Tabela larga por município: features do ciclo anterior + alvo do ciclo atual.",
        "pergunta": "Dá para prever a taxa do próximo ciclo e priorizar municípios em risco?",
        "particoes": None,
        "sql": """
            WITH marcos AS (
                SELECT MAX(ano) AS ano_alvo FROM silver_alfabetizacao_municipio
            ),
            atual AS (
                SELECT *
                FROM silver_alfabetizacao_municipio
                WHERE ano = (SELECT ano_alvo FROM marcos) AND rede = 3
            ),
            anterior AS (
                SELECT
                    id_municipio,
                    taxa_alfabetizacao      AS taxa_ciclo_anterior,
                    media_portugues         AS media_ciclo_anterior,
                    percentual_participacao AS participacao_ciclo_anterior,
                    taxa_uf                 AS taxa_uf_ciclo_anterior
                FROM silver_alfabetizacao_municipio
                WHERE ano = (SELECT ano_alvo - 1 FROM marcos) AND rede = 3
            ),
            microdados AS (
                SELECT
                    id_municipio,
                    COUNT(*) AS alunos_cadastrados,
                    SUM(CASE WHEN avaliado THEN 1 ELSE 0 END) AS alunos_avaliados,
                    ROUND(SUM(CASE WHEN avaliado THEN proficiencia_portugues * peso_aluno END)
                          / NULLIF(SUM(CASE WHEN avaliado THEN peso_aluno END), 0), 2)
                        AS proficiencia_media,
                    ROUND(STDDEV(CASE WHEN avaliado THEN proficiencia_portugues END), 2)
                        AS proficiencia_desvio
                FROM silver_aluno
                WHERE ano = (SELECT ano_alvo FROM marcos) AND rede = 3
                GROUP BY id_municipio
            )
            SELECT
                a.ano,
                a.id_municipio,
                a.nome_municipio,
                a.sigla_uf,
                a.nome_regiao,
                p.taxa_ciclo_anterior,
                p.media_ciclo_anterior,
                p.participacao_ciclo_anterior,
                p.taxa_uf_ciclo_anterior,
                a.percentual_participacao,
                a.taxa_uf,
                a.taxa_brasil_publica,
                a.meta_taxa_ano,
                a.meta_taxa_2030,
                a.gap_meta_2030,
                d.alunos_cadastrados,
                d.alunos_avaliados,
                d.proficiencia_media,
                d.proficiencia_desvio,
                a.taxa_alfabetizacao AS alvo_taxa_alfabetizacao,
                CASE WHEN a.atingiu_meta_ano THEN 1 ELSE 0 END AS alvo_atingiu_meta
            FROM atual a
            LEFT JOIN anterior p ON a.id_municipio = p.id_municipio
            LEFT JOIN microdados d ON a.id_municipio = d.id_municipio
        """,
    },

    # ------------------------------------------------------------------
    "gold_avaliacao_streaming": {
        "descricao": "Consolidação near-real-time das avaliações recebidas por streaming.",
        "pergunta": "O que está chegando agora e como se compara ao consolidado oficial?",
        "particoes": None,
        "opcional": True,
        "sql": """
            SELECT
                s.id_municipio,
                s.nome_municipio,
                s.sigla_uf,
                s.nome_regiao,
                s.rede_nome,
                COUNT(*) AS avaliacoes_recebidas,
                ROUND(100.0 * SUM(CASE WHEN s.alfabetizado THEN 1 ELSE 0 END) / COUNT(*), 2)
                    AS taxa_stream,
                ROUND(AVG(s.proficiencia_portugues), 2) AS proficiencia_media_stream,
                ROUND(AVG(s.latencia_ms), 1) AS latencia_media_ms,
                MIN(s.ts_evento) AS primeiro_evento,
                MAX(s.ts_evento) AS ultimo_evento
            FROM silver_avaliacao_stream s
            GROUP BY s.id_municipio, s.nome_municipio, s.sigla_uf, s.nome_regiao, s.rede_nome
        """,
    },
}

In [ ]:
# ============================================================
# ETAPA 3.2 — EXECUÇÃO DAS CONSULTAS E ESCRITA DA CAMADA GOLD
# ============================================================
from src.gold.consultas import CONSULTAS_GOLD
from src.utils.monitoring import PipelineMonitor

monitor = PipelineMonitor(pipeline="tc2-alfabetizacao-gold", run_id=RUN_ID,
                          output_dir=MONITOR, sla_etapa_segundos=180)

INVENTARIO_GOLD = []
gold = {}


def tamanho_diretorio(caminho):
    caminho = Path(caminho)
    if caminho.is_file():
        return caminho.stat().st_size
    return sum(f.stat().st_size for f in caminho.rglob("*") if f.is_file())


def formatar_bytes(n):
    n = float(n)
    for unidade in ("B", "KB", "MB", "GB"):
        if n < 1024 or unidade == "GB":
            return f"{n:.1f} {unidade}"
        n /= 1024


def gravar_gold(resultado, nome, particoes=None):
    """Grava uma tabela da Gold, no motor que estiver rodando."""
    destino = GOLD / nome
    if destino.exists():
        shutil.rmtree(destino)

    if MOTOR == "spark":
        if particoes:
            (resultado.write.mode("overwrite").option("compression", "snappy")
             .partitionBy(*particoes).parquet(str(destino)))
        else:
            # coalesce(1) para não gerar muitos arquivos pequenos
            (resultado.coalesce(1).write.mode("overwrite")
             .option("compression", "snappy").parquet(str(destino)))
        linhas = resultado.count()
    else:
        destino.mkdir(parents=True, exist_ok=True)
        from src.utils.lakeio import preparar_particoes
        if particoes:
            resultado = preparar_particoes(resultado, particoes)
            resultado.to_parquet(destino, engine="pyarrow", compression="snappy",
                                 partition_cols=particoes, index=False)
        else:
            resultado.to_parquet(destino / "part-00000.parquet", engine="pyarrow",
                                 compression="snappy", index=False)
        linhas = len(resultado)

    INVENTARIO_GOLD.append({
        "tabela": nome,
        "linhas": linhas,
        "particionada_por": ",".join(particoes) if particoes else "-",
        "bytes": tamanho_diretorio(destino),
        "caminho_s3": f"{S3_PREFIX['gold']}/{nome}/",
        "pergunta": CONSULTAS_GOLD[nome]["pergunta"],
    })
    return linhas


for nome, spec in CONSULTAS_GOLD.items():
    dependencia_ausente = (
        "silver_avaliacao_stream" in spec["sql"] and "silver_avaliacao_stream" not in views_registradas
    )
    if dependencia_ausente:
        log.warning("Consulta '%s' ignorada: depende de tabela ausente (execute o Notebook 02)", nome)
        continue

    with monitor.etapa(f"gold:{nome}", camada="gold") as metricas:
        try:
            resultado = executar_sql(spec["sql"])
            linhas = gravar_gold(resultado, nome, spec.get("particoes"))
            gold[nome] = resultado
            metricas["linhas_saida"] = linhas
            metricas["bytes_gerados"] = tamanho_diretorio(GOLD / nome)
            log.info("[GOLD] %-30s | %7d linhas | %s", nome, linhas,
                     formatar_bytes(tamanho_diretorio(GOLD / nome)))
        except Exception as exc:
            if spec.get("opcional"):
                log.warning("[GOLD] '%s' falhou e é opcional (%s) — seguindo", nome, exc)
                monitor.registrar_alerta("WARNING", f"Tabela Gold opcional '{nome}' não gerada: {exc}")
            else:
                raise

df_inventario_gold = pd.DataFrame(INVENTARIO_GOLD)
df_inventario_gold["tamanho"] = df_inventario_gold["bytes"].apply(formatar_bytes)
(GOLD / f"_inventario_{RUN_ID}.json").write_text(
    json.dumps(INVENTARIO_GOLD, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"\nGOLD — {len(df_inventario_gold)} tabelas | "
      f"{df_inventario_gold['linhas'].sum():,} linhas | "
      f"{formatar_bytes(df_inventario_gold['bytes'].sum())}".replace(",", "."))
display(df_inventario_gold[["tabela", "linhas", "particionada_por", "tamanho", "pergunta"]])

---
## Etapa 4 - Otimização

Três otimizações respondem pela maior parte da diferença de custo em Spark, e aqui elas são medidas.

Broadcast join: um join distribuído normal manda dados pela rede entre executores. Com um dos lados
pequeno, como `dim_municipio` com 5,5 mil linhas e `dim_uf` com 27, sai mais barato replicar a tabela
menor em todos os executores e evitar o shuffle. No plano físico isso aparece como
`BroadcastHashJoin` em vez de `SortMergeJoin`.

Plano físico: o `.explain()` mostra o que o Spark vai realmente executar, e é onde se confirma se o
partition pruning e o predicate pushdown funcionaram, pelas linhas `PartitionFilters` e
`PushedFilters`.

Cache: a Silver integrada alimenta 7 consultas Gold, e recalculá-la a cada uma é desperdício. Mas
cache ocupa memória do executor, então só compensa para dado reutilizado.

In [ ]:
# ============================================================
# ETAPA 4.1 — BROADCAST JOIN, PLANO FÍSICO E CACHE (medidos)
# ============================================================
if MOTOR == "spark":
    fatos = spark.table("silver_alfabetizacao_municipio")
    dimensao = spark.table("silver_dim_municipio").select(
        "id_municipio", "nome_microrregiao", "nome_regiao_imediata")

    # ── Sem broadcast (shuffle join) ──
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)   # desliga o automático
    inicio = time.perf_counter()
    sem_broadcast = fatos.join(dimensao, on="id_municipio", how="left")
    linhas_sem = sem_broadcast.count()
    tempo_sem = time.perf_counter() - inicio

    # ── Com broadcast explícito ──
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)
    inicio = time.perf_counter()
    com_broadcast = fatos.join(broadcast(dimensao), on="id_municipio", how="left")
    linhas_com = com_broadcast.count()
    tempo_com = time.perf_counter() - inicio

    print("COMPARAÇÃO DE ESTRATÉGIA DE JOIN")
    display(pd.DataFrame([
        {"estrategia": "shuffle join (sem broadcast)", "linhas": linhas_sem,
         "tempo_s": round(tempo_sem, 2)},
        {"estrategia": "broadcast join", "linhas": linhas_com,
         "tempo_s": round(tempo_com, 2)},
        {"estrategia": "ganho", "linhas": "-",
         "tempo_s": f"{(1 - tempo_com / max(tempo_sem, 1e-9)) * 100:.0f}%"},
    ]))

    print("\nPLANO FÍSICO COM BROADCAST (procure por 'BroadcastHashJoin'):")
    com_broadcast.explain(mode="formatted")

    print("\nPLANO DE LEITURA COM FILTRO DE PARTIÇÃO "
          "(procure por 'PartitionFilters' e 'PushedFilters'):")
    spark.table("silver_alfabetizacao_municipio").filter("ano = 2024 AND rede = 3").explain()

    # ── Cache: a Silver integrada alimenta 7 consultas Gold ──
    fatos.cache()
    fatos.count()          # materializa o cache
    print(f"\nDataFrame em cache: {fatos.storageLevel}")
else:
    print("Otimizações específicas de Spark (broadcast join, plano físico, cache) exigem PySpark.")
    print("Execute este notebook no Google Colab para ver esta etapa com números medidos.")
    print("A lógica da camada Gold é idêntica nos dois motores — o que muda é o executor.")

---
## Etapa 5 - FinOps

FinOps aqui não é escolher a instância mais barata, e sim entender o que é cobrado e desenhar a
pipeline em torno disso.

| Serviço | Unidade cobrada | Alavanca de economia |
|---|---|---|
| S3 | GB armazenado por mês | formato comprimido e ciclo de vida por camada |
| Athena | TB escaneado | particionamento, projeção de colunas e Parquet |
| Glue | DPU-hora | menos shuffle, menos re-leitura, job dimensionado |
| MSK | hora de broker e GB | streaming só na janela em que há evento |

A alavanca dominante é a segunda, já que o Athena cobra por byte lido e não por linha retornada. As
células abaixo medem isso nos próprios arquivos do projeto: a mesma pergunta respondida de quatro
formas, com os bytes lidos em cada uma.

In [ ]:
# ============================================================
# ETAPA 5.1 — COMPRESSÃO: CSV vs PARQUET, MEDIDO
# ============================================================
import tempfile
import pyarrow.parquet as pq

bytes_landing = sum(f.stat().st_size for f in LANDING.glob("*.csv"))
bytes_bronze  = tamanho_diretorio(BRONZE)
bytes_silver  = tamanho_diretorio(SILVER)
bytes_gold    = tamanho_diretorio(GOLD)

volumes = pd.DataFrame([
    {"artefato": "Landing (CSV bruto, 5 arquivos)", "bytes": bytes_landing},
    {"artefato": "Bronze (Parquet+Snappy, 8 entidades)", "bytes": bytes_bronze},
    {"artefato": "Silver (Parquet+Snappy, 11 tabelas)", "bytes": bytes_silver},
    {"artefato": "Gold (Parquet+Snappy, 8 tabelas)", "bytes": bytes_gold},
])
volumes["tamanho"] = volumes["bytes"].apply(formatar_bytes)
print("VOLUME POR CAMADA")
display(volumes[["artefato", "tamanho"]])
linhas_aluno = len(ler_parquet(BRONZE / "aluno"))
print(f"Obs.: a Bronze é maior que o landing porque inclui {linhas_aluno:,} microdados de aluno"
      .replace(",", "."))
print("      e as colunas de linhagem — não é uma comparação de formato.\n")

# ── Comparação justa: MESMOS dados, MESMAS colunas, formatos diferentes ──
amostra = ler_parquet(SILVER / "fato_indicador_municipio")
amostra = amostra[[c for c in amostra.columns if not c.startswith("_")]]

temporario = Path(tempfile.mkdtemp())
amostra.to_csv(temporario / "amostra.csv", index=False)
amostra.to_csv(temporario / "amostra.csv.gz", index=False, compression="gzip")
amostra.to_parquet(temporario / "amostra.parquet", engine="pyarrow", compression="snappy")

formatos = pd.DataFrame([
    {"formato": "CSV", "bytes": (temporario / "amostra.csv").stat().st_size},
    {"formato": "CSV + gzip", "bytes": (temporario / "amostra.csv.gz").stat().st_size},
    {"formato": "Parquet + Snappy", "bytes": (temporario / "amostra.parquet").stat().st_size},
])
base_csv = formatos.loc[formatos["formato"] == "CSV", "bytes"].iloc[0]
formatos["tamanho"] = formatos["bytes"].apply(formatar_bytes)
formatos["reducao_vs_csv"] = ((1 - formatos["bytes"] / base_csv) * 100).round(1).astype(str) + "%"

print(f"MESMOS DADOS ({len(amostra):,} linhas x {len(amostra.columns)} colunas), "
      f"TRÊS FORMATOS".replace(",", "."))
display(formatos[["formato", "tamanho", "reducao_vs_csv"]])
print("\nO Parquet ganha do CSV compactado não pelo tamanho em disco, mas por ser COLUNAR:")
print("é isso que permite ler 4 colunas sem tocar nas outras — o que a próxima célula mede.")

In [ ]:
# ============================================================
# ETAPA 5.2 — BYTES ESCANEADOS: A MESMA PERGUNTA, TRÊS CUSTOS
# ============================================================
# Pergunta: "taxa de alfabetização por município na rede Municipal em 2024?"
# São 4 colunas de 1 ano. A medição abaixo mostra quantos bytes cada estratégia
# obriga a ler, que é o que o Athena cobra.

COLUNAS_NECESSARIAS = {"id_municipio", "nome_municipio", "taxa_alfabetizacao", "rede_nome"}
TABELA_ALVO = GOLD / "gold_indicador_municipio"
ANO_ALVO = "2024"


def bytes_das_colunas(caminho, colunas=None):
    """Bytes realmente lidos ao projetar só algumas colunas.

    Lê os metadados do Parquet (tamanho por coluna em cada row group), que é o
    mesmo mecanismo que faz o Athena cobrar só pelo que foi lido.
    """
    total = 0
    for arquivo in sorted(Path(caminho).rglob("*.parquet")):
        metadados = pq.ParquetFile(arquivo).metadata
        nomes = [metadados.schema.column(i).name for i in range(metadados.num_columns)]
        for grupo in range(metadados.num_row_groups):
            for indice, nome in enumerate(nomes):
                if colunas is None or nome in colunas:
                    total += metadados.row_group(grupo).column(indice).total_compressed_size
    return total


# Baseline "sem data lake": o CSV original, varrido inteiro a cada consulta.
csv_indicador = LANDING / "br_inep_avaliacao_alfabetizacao_municipio.csv"

bytes_csv = csv_indicador.stat().st_size                                  # varre o arquivo inteiro
bytes_parquet_total = bytes_das_colunas(TABELA_ALVO)                      # todas as partições/colunas
particao_alvo = TABELA_ALVO / f"ano={ANO_ALVO}"
bytes_particao = bytes_das_colunas(particao_alvo)                         # só 2024
bytes_otimizado = bytes_das_colunas(particao_alvo, COLUNAS_NECESSARIAS)   # 2024 + 4 colunas

PRECO_ATHENA_POR_TB = 5.00
CONSULTAS_POR_MES = 500


def custo_athena(bytes_lidos, consultas=CONSULTAS_POR_MES):
    return bytes_lidos * consultas / (1024 ** 4) * PRECO_ATHENA_POR_TB


estrategias = pd.DataFrame([
    {"estrategia": "1. CSV — varredura completa", "bytes": bytes_csv},
    {"estrategia": "2. Parquet sem particionamento", "bytes": bytes_parquet_total},
    {"estrategia": "3. Parquet + partição (ano=2024)", "bytes": bytes_particao},
    {"estrategia": "4. Parquet + partição + projeção de colunas", "bytes": bytes_otimizado},
])
estrategias["lido"] = estrategias["bytes"].apply(formatar_bytes)
estrategias["vs_csv"] = (estrategias["bytes"] / bytes_csv * 100).round(1).astype(str) + "%"
estrategias[f"custo_athena_{CONSULTAS_POR_MES}_consultas_usd"] = (
    estrategias["bytes"].apply(custo_athena).round(4))

print("BYTES LIDOS PARA RESPONDER À MESMA PERGUNTA\n")
display(estrategias[["estrategia", "lido", "vs_csv",
                     f"custo_athena_{CONSULTAS_POR_MES}_consultas_usd"]])

fator = bytes_csv / max(bytes_otimizado, 1)
print(f"\nA estratégia 4 lê {fator:.0f}x menos bytes que a estratégia 1 — mesmo resultado.")
print("No Athena, isso é exatamente a razão entre as duas contas no fim do mês.")

In [ ]:
# ============================================================
# ETAPA 5.3 — ESTIMATIVA DE CUSTO MENSAL DA ARQUITETURA
# ============================================================
# Preços de referência (us-east-1). São valores de tabela pública e devem ser
# reconferidos no AWS Pricing Calculator antes de virar orçamento.
PRECOS = {
    "s3_standard_gb_mes":        0.023,
    "s3_standard_ia_gb_mes":     0.0125,
    "s3_glacier_ir_gb_mes":      0.004,
    "athena_por_tb":             5.00,
    "glue_dpu_hora":             0.44,
    "msk_serverless_hora":       0.75,
    "lambda_por_milhao_req":     0.20,
    "cloudwatch_metrica_mes":    0.30,
}

# Premissas do cenário de produção. O dado oficial é anual, mas a operação não é:
# revisões de meta e correções chegam ao longo do ano.
PREMISSAS = {
    "fator_volume_producao": 50,   # microdados completos do Saeb vs. amostra deste protótipo
    "execucoes_batch_mes": 4,      # 1 por semana (captura republicações da fonte)
    "dpus_por_execucao": 2,
    "horas_por_execucao": 0.25,
    "semanas_streaming_ano": 6,    # janela de aplicação da avaliação
    "consultas_athena_mes": 500,
    "metricas_cloudwatch": 10,
}

gb_bronze = bytes_bronze * PREMISSAS["fator_volume_producao"] / (1024 ** 3)
gb_silver = bytes_silver * PREMISSAS["fator_volume_producao"] / (1024 ** 3)
gb_gold   = bytes_gold   * PREMISSAS["fator_volume_producao"] / (1024 ** 3)

custo_glue = (PREMISSAS["execucoes_batch_mes"] * PREMISSAS["dpus_por_execucao"]
              * PREMISSAS["horas_por_execucao"] * PRECOS["glue_dpu_hora"])
custo_athena_mes = custo_athena(bytes_otimizado * PREMISSAS["fator_volume_producao"],
                                PREMISSAS["consultas_athena_mes"])
horas_msk_mes = PREMISSAS["semanas_streaming_ano"] * 7 * 24 / 12
custo_msk = horas_msk_mes * PRECOS["msk_serverless_hora"]

componentes = pd.DataFrame([
    {"componente": "S3 Bronze (Glacier IR após 90 dias)",
     "dimensionamento": f"{gb_bronze:.2f} GB",
     "custo_mes_usd": gb_bronze * PRECOS["s3_glacier_ir_gb_mes"]},
    {"componente": "S3 Silver (Standard-IA após 30 dias)",
     "dimensionamento": f"{gb_silver:.2f} GB",
     "custo_mes_usd": gb_silver * PRECOS["s3_standard_ia_gb_mes"]},
    {"componente": "S3 Gold (Standard — sempre quente)",
     "dimensionamento": f"{gb_gold:.2f} GB",
     "custo_mes_usd": gb_gold * PRECOS["s3_standard_gb_mes"]},
    {"componente": "AWS Glue (ETL batch)",
     "dimensionamento": f"{PREMISSAS['execucoes_batch_mes']}x {PREMISSAS['dpus_por_execucao']} DPU "
                        f"x {PREMISSAS['horas_por_execucao']}h",
     "custo_mes_usd": custo_glue},
    {"componente": "Amazon Athena (consultas analíticas)",
     "dimensionamento": f"{PREMISSAS['consultas_athena_mes']} consultas otimizadas",
     "custo_mes_usd": custo_athena_mes},
    {"componente": "Amazon MSK Serverless (janela de aplicação)",
     "dimensionamento": f"{horas_msk_mes:.0f} h/mês (média anual)",
     "custo_mes_usd": custo_msk},
    {"componente": "CloudWatch (métricas e alarmes)",
     "dimensionamento": f"{PREMISSAS['metricas_cloudwatch']} métricas",
     "custo_mes_usd": PREMISSAS["metricas_cloudwatch"] * PRECOS["cloudwatch_metrica_mes"]},
])
componentes["custo_mes_usd"] = componentes["custo_mes_usd"].round(4)
total_mes = componentes["custo_mes_usd"].sum()

print("ESTIMATIVA DE CUSTO MENSAL — ARQUITETURA COMPLETA\n")
display(componentes)
print(f"TOTAL ESTIMADO: US$ {total_mes:.2f}/mês  (~US$ {total_mes * 12:.2f}/ano)")

# ── O contraste que justifica a arquitetura híbrida ──
msk_sempre_ligado = 730 * PRECOS["msk_serverless_hora"]
economia_anual = (msk_sempre_ligado - custo_msk) * 12
print(f"\nSe o streaming ficasse ligado o ano inteiro: US$ {msk_sempre_ligado:.2f}/mês só de MSK.")
print(f"Ligar o broker apenas na janela de aplicação economiza "
      f"US$ {economia_anual:,.2f}/ano".replace(",", "."))
print("É a maior decisão de FinOps do projeto — e ela é arquitetural, não de configuração.")

---
### 5.4 - Resumo das decisões de FinOps

| Decisão | Efeito |
|---|---|
| Parquet com Snappy em todas as camadas | formato colunar, lê só as colunas necessárias |
| Particionamento por `ano` | consultas leem só o ano pedido |
| Projeção de colunas, nunca `SELECT *` | Athena lê apenas o que entra no resultado |
| Streaming só na janela de aplicação | maior economia isolada do projeto |
| Ciclo de vida por camada, Gold quente e Bronze Glacier | menos custo no histórico |
| `shuffle.partitions = 8`, AQE e broadcast join | menos arquivos pequenos e menos tempo de DPU |
| `partitionOverwriteMode = dynamic` | reprocessar 1 ano não reescreve o histórico |

As três primeiras não são só economia: menos bytes lidos também significa consulta mais rápida.

---

## Etapa 6 - Materialização na AWS

As células a seguir geram os artefatos de deploy: os três jobs do Glue, o DDL do Athena, o
processador de streaming em Lambda e a orquestração em Step Functions. São os mesmos passos
executados aqui, mudando os caminhos (`s3://` no lugar de `data/lake/`) e o contexto de execução.

```
EventBridge (agenda) → Step Functions → Glue bronze → Glue silver → Glue gold → Crawler → Athena
                              │              │             │            │
                              └─ SNS (falha) └─ DQ gate ───┴── DQ gate ─┘
```

In [ ]:
%%writefile cloud/aws/glue/etl_bronze.py
"""AWS Glue Job — Camada BRONZE da pipeline de alfabetização.

Ingere as fontes do desafio e grava a Bronze em Parquet particionado, preservando
o dado bruto e anexando linhagem. Espelho do Notebook 01, Etapa 2.

Parâmetros do job (Job details -> Advanced properties -> Job parameters):
    --BUCKET_LANDING   420411424817-tc2-landing
    --BUCKET_SOR       420411424817-tc2-data-sor
    --ENTIDADES        indicador_municipio,indicador_uf,meta_brasil,meta_uf,meta_municipio
    --API_IBGE         https://servicodados.ibge.gov.br/api/v1/localidades

Recomendação de execução (FinOps):
    Worker type G.1X | Number of workers 2 | Job bookmark: enable | Auto scaling: on
"""
import sys
import logging
from datetime import datetime, timezone

import requests
import pandas as pd

from awsglue.utils import getResolvedOptions
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.context import SparkContext
from pyspark.sql import functions as F

# ============================================================
# LOGGING
# ============================================================
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-8s | %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%SZ",
)
log = logging.getLogger(__name__)

# ============================================================
# PARÂMETROS
# ============================================================
args = getResolvedOptions(sys.argv, [
    "JOB_NAME", "BUCKET_LANDING", "BUCKET_SOR", "ENTIDADES", "API_IBGE",
])

sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args["JOB_NAME"], args)

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
spark.conf.set("spark.sql.parquet.compression.codec", "snappy")
spark.sparkContext.setLogLevel("WARN")

JOB_NAME       = args["JOB_NAME"]
BUCKET_LANDING = args["BUCKET_LANDING"]
BUCKET_SOR     = args["BUCKET_SOR"]
ENTIDADES      = [e.strip() for e in args["ENTIDADES"].split(",") if e.strip()]
API_IBGE       = args["API_IBGE"]

INGESTION_TS   = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
INGESTION_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")
ANO, MES, DIA  = INGESTION_DATE.split("-")

ARQUIVOS = {
    "indicador_municipio": "br_inep_avaliacao_alfabetizacao_municipio.csv",
    "indicador_uf":        "br_inep_avaliacao_alfabetizacao_uf.csv",
    "meta_brasil":         "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv",
    "meta_uf":             "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv",
    "meta_municipio":      "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv",
}

# Regras mínimas de aceitação por entidade — o job falha (e alerta) se não passarem.
CHECKS = {
    "indicador_municipio": [("min_count", None, 10000), ("not_null", "id_municipio", None),
                            ("not_null", "taxa_alfabetizacao", None)],
    "indicador_uf":        [("min_count", None, 100), ("not_null", "sigla_uf", None)],
    "meta_brasil":         [("min_count", None, 1), ("not_null", "ano", None)],
    "meta_uf":             [("min_count", None, 20), ("not_null", "sigla_uf", None)],
    "meta_municipio":      [("min_count", None, 10000), ("not_null", "id_municipio", None)],
    "dim_uf_ibge":         [("min_count", None, 27), ("unique", "id", None)],
    "dim_municipio_ibge":  [("min_count", None, 5500), ("unique", "id", None)],
}

log.info("=" * 70)
log.info("JOB      : %s", JOB_NAME)
log.info("ENTIDADES: %s", ENTIDADES)
log.info("DESTINO  : s3://%s/bronze/", BUCKET_SOR)
log.info("=" * 70)


# ============================================================
# FUNÇÕES
# ============================================================
def com_linhagem(df, entidade, origem, formato):
    """Anexa as colunas de linhagem e as partições físicas de ingestão."""
    return (df
            .withColumn("_ingestion_timestamp", F.lit(INGESTION_TS))
            .withColumn("_ingestion_date", F.lit(INGESTION_DATE))
            .withColumn("_source", F.lit(origem))
            .withColumn("_source_format", F.lit(formato))
            .withColumn("_entity", F.lit(entidade))
            .withColumn("_pipeline_run_id", F.lit(JOB_NAME + "_" + INGESTION_TS))
            .withColumn("_record_hash", F.sha2(F.concat_ws("||", *df.columns), 256))
            .withColumn("ano_ingestao", F.lit(ANO))
            .withColumn("mes_ingestao", F.lit(MES))
            .withColumn("dia_ingestao", F.lit(DIA)))


def checar_qualidade(df, entidade):
    """Quality gate da Bronze. Falha crítica interrompe o job."""
    regras = CHECKS.get(entidade, [])
    if not regras:
        log.warning("[DQ:BRONZE] Sem regras para '%s' — pulando", entidade)
        return

    falhas = []
    for tipo, coluna, valor in regras:
        if tipo == "min_count":
            total = df.count()
            ok, detalhe = total >= valor, f"contagem={total} minimo={valor}"
        elif tipo == "not_null":
            nulos = df.filter(F.col(coluna).isNull()).count()
            ok, detalhe = nulos == 0, f"{nulos} nulos em {coluna}"
        elif tipo == "unique":
            duplicadas = df.count() - df.select(coluna).distinct().count()
            ok, detalhe = duplicadas == 0, f"{duplicadas} duplicatas em {coluna}"
        else:
            ok, detalhe = True, "tipo de check desconhecido"

        log.info("[DQ:BRONZE] %-4s | %-10s | %s", "PASS" if ok else "FAIL", tipo, detalhe)
        if not ok:
            falhas.append(f"{tipo}:{coluna}:{detalhe}")

    if falhas:
        raise Exception(f"[QUALITY GATE] {entidade}: " + "; ".join(falhas))


def gravar(df, entidade):
    destino = f"s3://{BUCKET_SOR}/bronze/{entidade}"
    (df.write
       .mode("overwrite")
       .option("compression", "snappy")
       .partitionBy("ano_ingestao", "mes_ingestao", "dia_ingestao")
       .parquet(destino))
    log.info("[BRONZE] %s | %d linhas -> %s", entidade, df.count(), destino)
    return destino


# ============================================================
# EXECUÇÃO — CSVs DA LANDING
# ============================================================
for entidade in ENTIDADES:
    arquivo = ARQUIVOS.get(entidade)
    if not arquivo:
        log.warning("Entidade desconhecida, ignorando: %s", entidade)
        continue

    origem = f"s3://{BUCKET_LANDING}/basedosdados/{arquivo}"
    log.info("[INGESTAO] Lendo %s", origem)
    bruto = (spark.read
             .option("header", "true")
             .option("inferSchema", "true")
             .csv(origem))

    bronze = com_linhagem(bruto, entidade, origem, "csv")
    checar_qualidade(bronze, entidade)
    gravar(bronze, entidade)

# ============================================================
# EXECUÇÃO — API DE LOCALIDADES DO IBGE
# ============================================================
for entidade, rota in (("dim_uf_ibge", "estados"), ("dim_municipio_ibge", "municipios")):
    url = f"{API_IBGE}/{rota}"
    log.info("[INGESTAO] GET %s", url)
    resposta = requests.get(url, timeout=60)
    resposta.raise_for_status()

    achatado = pd.json_normalize(resposta.json())
    achatado.columns = [c.replace(".", "_").replace("-", "_") for c in achatado.columns]
    bronze = com_linhagem(spark.createDataFrame(achatado), entidade, url, "json")
    checar_qualidade(bronze, entidade)
    gravar(bronze, entidade)

log.info("=" * 70)
log.info("BRONZE CONCLUIDA | proxima etapa: etl_silver")
log.info("=" * 70)

job.commit()

In [ ]:
%%writefile cloud/aws/glue/etl_silver.py
"""AWS Glue Job — Camada SILVER da pipeline de alfabetização.

Padroniza tipos, decodifica domínios, normaliza chaves, deduplica pelo grão e
**integra** indicador + metas + território. Espelho do Notebook 01, Etapa 4.

Parâmetros do job:
    --BUCKET_SOR   420411424817-tc2-data-sor
    --BUCKET_SOT   420411424817-tc2-data-sot
"""
import sys
import logging
from datetime import datetime, timezone

from awsglue.utils import getResolvedOptions
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.context import SparkContext
from pyspark.sql import functions as F
from pyspark.sql.functions import broadcast
from pyspark.sql.window import Window

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)-8s | %(message)s",
                    datefmt="%Y-%m-%dT%H:%M:%SZ")
log = logging.getLogger(__name__)

args = getResolvedOptions(sys.argv, ["JOB_NAME", "BUCKET_SOR", "BUCKET_SOT"])

sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args["JOB_NAME"], args)

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "16")
spark.sparkContext.setLogLevel("WARN")

BUCKET_SOR = args["BUCKET_SOR"]
BUCKET_SOT = args["BUCKET_SOT"]
RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

PONTO_CORTE_SAEB = 743
META_NACIONAL_2030 = 80.0
DIM_REDE = {0: "Total", 1: "Federal", 2: "Estadual", 3: "Municipal", 4: "Privada", 5: "Publica"}
REDE_TEXTO = {"municipal": 3, "publica": 5, "pública": 5, "estadual": 2, "federal": 1, "privada": 4}


def ler_bronze(entidade):
    return spark.read.parquet(f"s3://{BUCKET_SOR}/bronze/{entidade}")


def gravar_silver(df, tabela, particoes=None):
    destino = f"s3://{BUCKET_SOT}/silver/{tabela}"
    escritor = (df.withColumn("_silver_timestamp", F.lit(RUN_TS))
                  .write.mode("overwrite").option("compression", "snappy"))
    if particoes:
        escritor = escritor.partitionBy(*particoes)
    escritor.parquet(destino)
    log.info("[SILVER] %-32s | %8d linhas -> %s", tabela, df.count(), destino)
    return destino


def mapear_rede(coluna):
    """Traduz o código numérico da rede para nome, via CASE WHEN encadeado."""
    expressao = F.lit(None).cast("string")
    for codigo, nome in DIM_REDE.items():
        expressao = F.when(coluna == codigo, F.lit(nome)).otherwise(expressao)
    return expressao


# ============================================================
# DIMENSÕES TERRITORIAIS
# ============================================================
dim_uf = (ler_bronze("dim_uf_ibge")
          .select(F.col("id").cast("int").alias("id_uf"),
                  F.upper(F.trim(F.col("sigla"))).alias("sigla_uf"),
                  F.trim(F.col("nome")).alias("nome_uf"),
                  F.col("regiao_sigla").alias("sigla_regiao"),
                  F.col("regiao_nome").alias("nome_regiao"))
          .dropDuplicates(["id_uf"]))
gravar_silver(dim_uf, "dim_uf")

dim_municipio = (ler_bronze("dim_municipio_ibge")
                 .select(F.col("id").cast("int").alias("id_municipio"),
                         F.trim(F.col("nome")).alias("nome_municipio"),
                         F.col("microrregiao_nome").alias("nome_microrregiao"),
                         F.col("microrregiao_mesorregiao_nome").alias("nome_mesorregiao"))
                 .withColumn("id_uf", (F.col("id_municipio") / 100000).cast("int"))
                 .join(broadcast(dim_uf), on="id_uf", how="left")
                 .filter(F.col("sigla_uf").isNotNull()))
gravar_silver(dim_municipio, "dim_municipio")

# ============================================================
# FATOS DO INDICADOR
# ============================================================
COLUNAS_NIVEL = [f"proporcao_aluno_nivel_{i}" for i in range(9)]

indicador_municipio = (ler_bronze("indicador_municipio")
    .withColumn("ano", F.col("ano").cast("int"))
    .withColumn("id_municipio", F.col("id_municipio").cast("int"))
    .withColumn("serie", F.col("serie").cast("int"))
    .withColumn("rede", F.col("rede").cast("int"))
    .withColumn("taxa_alfabetizacao", F.round(F.col("taxa_alfabetizacao").cast("double"), 2))
    .withColumn("media_portugues", F.round(F.col("media_portugues").cast("double"), 2))
    .filter(F.col("ano").isNotNull()
            & F.col("id_municipio").between(1000000, 9999999)
            & F.col("taxa_alfabetizacao").between(0, 100))
    .withColumn("rede_nome", mapear_rede(F.col("rede")))
    .dropDuplicates(["ano", "id_municipio", "serie", "rede"]))

fato_indicador_municipio = (indicador_municipio
    .select("ano", "id_municipio", "serie", "rede", "rede_nome",
            "taxa_alfabetizacao", "media_portugues")
    .withColumn("id_uf", (F.col("id_municipio") / 100000).cast("int"))
    .withColumn("alcancou_meta_2030", F.col("taxa_alfabetizacao") >= F.lit(META_NACIONAL_2030)))
gravar_silver(fato_indicador_municipio, "fato_indicador_municipio", ["ano"])

fato_indicador_uf = (ler_bronze("indicador_uf")
    .withColumn("ano", F.col("ano").cast("int"))
    .withColumn("rede", F.col("rede").cast("int"))
    .withColumn("sigla_uf", F.upper(F.trim(F.col("sigla_uf"))))
    .withColumn("taxa_alfabetizacao", F.round(F.col("taxa_alfabetizacao").cast("double"), 2))
    .withColumn("rede_nome", mapear_rede(F.col("rede")))
    .dropDuplicates(["ano", "sigla_uf", "serie", "rede"])
    .join(broadcast(dim_uf.select("sigla_uf", "id_uf", "nome_uf", "nome_regiao")),
          on="sigla_uf", how="left"))
gravar_silver(fato_indicador_uf, "fato_indicador_uf", ["ano"])

# Normalização: 9 colunas de proporção viram uma tabela longa
distribuicao = None
for nivel, coluna in enumerate(COLUNAS_NIVEL):
    parcial = (indicador_municipio
               .select("ano", "id_municipio", "rede", "rede_nome",
                       F.col(coluna).cast("double").alias("proporcao_alunos"))
               .withColumn("nivel", F.lit(nivel))
               .filter(F.col("proporcao_alunos").isNotNull()))
    distribuicao = parcial if distribuicao is None else distribuicao.unionByName(parcial)
gravar_silver(distribuicao, "fato_distribuicao_nivel", ["ano"])


# ============================================================
# TRAJETÓRIAS DE META (colunas por ano -> linhas)
# ============================================================
def trajetoria_metas(df, chaves):
    colunas_meta = [c for c in df.columns if c.startswith("meta_alfabetizacao_")]
    empilhados = None
    for coluna in colunas_meta:
        ano_meta = int(coluna.split("_")[-1])
        parcial = (df.select(*chaves, "rede", "ano",
                             F.col(coluna).cast("double").alias("meta_taxa"))
                     .withColumn("ano_meta", F.lit(ano_meta))
                     .filter(F.col("meta_taxa").isNotNull()))
        empilhados = parcial if empilhados is None else empilhados.unionByName(parcial)

    # Revisão vigente: a declarada no ano de referência mais recente
    janela = Window.partitionBy(*chaves, "rede", "ano_meta").orderBy(F.col("ano").desc())
    return (empilhados
            .withColumn("_ordem", F.row_number().over(janela))
            .filter(F.col("_ordem") == 1)
            .drop("_ordem", "ano"))


def normalizar_rede_texto(df):
    expressao = F.lit(None).cast("int")
    for texto, codigo in REDE_TEXTO.items():
        expressao = F.when(F.lower(F.trim(F.col("rede"))) == texto,
                           F.lit(codigo)).otherwise(expressao)
    return df.withColumn("rede", expressao)


meta_municipio = normalizar_rede_texto(
    ler_bronze("meta_municipio").withColumn("id_municipio", F.col("id_municipio").cast("int")))
gravar_silver(trajetoria_metas(meta_municipio, ["id_municipio"]), "dim_meta_municipio")

meta_uf = normalizar_rede_texto(
    ler_bronze("meta_uf").withColumn("sigla_uf", F.upper(F.trim(F.col("sigla_uf")))))
gravar_silver(trajetoria_metas(meta_uf, ["sigla_uf"]), "dim_meta_uf")

# ============================================================
# TABELA INTEGRADA
# ============================================================
metas_municipio = spark.read.parquet(f"s3://{BUCKET_SOT}/silver/dim_meta_municipio")
meta_2030 = (metas_municipio.filter(F.col("ano_meta") == 2030)
             .select("id_municipio", "rede", F.col("meta_taxa").alias("meta_taxa_2030")))
contexto_uf = (fato_indicador_uf
               .select("ano", "sigla_uf", "rede", F.col("taxa_alfabetizacao").alias("taxa_uf")))

integrada = (fato_indicador_municipio
    .join(broadcast(dim_municipio.select("id_municipio", "nome_municipio", "sigla_uf",
                                         "nome_uf", "nome_regiao", "nome_mesorregiao")),
          on="id_municipio", how="left")
    .join(metas_municipio.select("id_municipio", "rede",
                                 F.col("ano_meta").alias("ano"),
                                 F.col("meta_taxa").alias("meta_taxa_ano")),
          on=["id_municipio", "rede", "ano"], how="left")
    .join(broadcast(meta_2030), on=["id_municipio", "rede"], how="left")
    .join(contexto_uf, on=["ano", "sigla_uf", "rede"], how="left")
    .withColumn("meta_taxa_2030", F.coalesce(F.col("meta_taxa_2030"), F.lit(META_NACIONAL_2030)))
    .withColumn("gap_meta_ano", F.round(F.col("taxa_alfabetizacao") - F.col("meta_taxa_ano"), 2))
    .withColumn("atingiu_meta_ano", F.col("gap_meta_ano") >= 0)
    .withColumn("gap_meta_2030", F.round(F.col("meta_taxa_2030") - F.col("taxa_alfabetizacao"), 2))
    .withColumn("dif_vs_uf", F.round(F.col("taxa_alfabetizacao") - F.col("taxa_uf"), 2)))

# ── Quality gate da Silver ──
total = integrada.count()
duplicadas = total - integrada.dropDuplicates(["ano", "id_municipio", "rede"]).count()
sem_territorio = integrada.filter(F.col("sigla_uf").isNull()).count()
log.info("[DQ:SILVER] linhas=%d duplicadas=%d sem_territorio=%d", total, duplicadas, sem_territorio)
if duplicadas > 0 or sem_territorio > 0:
    raise Exception(f"[QUALITY GATE] Silver integrada: {duplicadas} duplicatas, "
                    f"{sem_territorio} linhas sem território")

gravar_silver(integrada, "fato_alfabetizacao_municipio", ["ano"])

log.info("=" * 70)
log.info("SILVER CONCLUIDA | proxima etapa: etl_gold")
log.info("=" * 70)

job.commit()

In [ ]:
%%writefile cloud/aws/glue/etl_gold.py
"""AWS Glue Job — Camada GOLD da pipeline de alfabetização.

Executa as MESMAS consultas SQL do Notebook 03 (`src/gold/consultas.py`), lendo a
Silver do bucket SOT e gravando a Gold no bucket SPEC.

Parâmetros do job:
    --BUCKET_SOT    420411424817-tc2-data-sot
    --BUCKET_SPEC   420411424817-tc2-data-spec
    --TABELAS       todas            (ou lista separada por vírgula)

Dependência:
    Job details -> Advanced properties -> Python library path:
        s3://<bucket-artefatos>/libs/consultas.py

    Esse arquivo é o próprio `src/gold/consultas.py` versionado no repositório —
    a regra de negócio da Gold tem uma única fonte da verdade.
"""
import sys
import json
import logging
from datetime import datetime, timezone

from awsglue.utils import getResolvedOptions
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.context import SparkContext

from consultas import CONSULTAS_GOLD

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s | %(levelname)-8s | %(message)s",
                    datefmt="%Y-%m-%dT%H:%M:%SZ")
log = logging.getLogger(__name__)

args = getResolvedOptions(sys.argv, ["JOB_NAME", "BUCKET_SOT", "BUCKET_SPEC", "TABELAS"])

sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)
job.init(args["JOB_NAME"], args)

spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "16")
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
spark.conf.set("spark.sql.parquet.compression.codec", "snappy")
spark.sparkContext.setLogLevel("WARN")

BUCKET_SOT  = args["BUCKET_SOT"]
BUCKET_SPEC = args["BUCKET_SPEC"]
SELECAO     = args["TABELAS"]
RUN_TS      = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

VIEWS_SILVER = {
    "silver_alfabetizacao_municipio": "fato_alfabetizacao_municipio",
    "silver_indicador_uf":            "fato_indicador_uf",
    "silver_dim_municipio":           "dim_municipio",
    "silver_dim_uf":                  "dim_uf",
    "silver_meta_municipio":          "dim_meta_municipio",
    "silver_meta_uf":                 "dim_meta_uf",
    "silver_distribuicao_nivel":      "fato_distribuicao_nivel",
    "silver_aluno":                   "fato_aluno",
    "silver_avaliacao_stream":        "fato_avaliacao_stream",
}

# ============================================================
# REGISTRO DAS VIEWS
# ============================================================
registradas = set()
for view, tabela in VIEWS_SILVER.items():
    caminho = f"s3://{BUCKET_SOT}/silver/{tabela}"
    try:
        spark.read.parquet(caminho).createOrReplaceTempView(view)
        registradas.add(view)
        log.info("[GOLD] View registrada: %s", view)
    except Exception as exc:
        log.warning("[GOLD] Tabela Silver ausente (%s): %s", tabela, exc)

# ============================================================
# EXECUÇÃO DAS CONSULTAS
# ============================================================
alvos = (list(CONSULTAS_GOLD) if SELECAO.strip().lower() == "todas"
         else [t.strip() for t in SELECAO.split(",") if t.strip()])

metricas = []
for nome in alvos:
    spec = CONSULTAS_GOLD[nome]

    if "silver_avaliacao_stream" in spec["sql"] and "silver_avaliacao_stream" not in registradas:
        log.warning("[GOLD] %s ignorada: depende do stream, ainda não disponível", nome)
        continue

    inicio = datetime.now(timezone.utc)
    try:
        resultado = spark.sql(spec["sql"])
        destino = f"s3://{BUCKET_SPEC}/gold/{nome}"
        escritor = resultado.write.mode("overwrite").option("compression", "snappy")
        if spec.get("particoes"):
            escritor = escritor.partitionBy(*spec["particoes"])
        escritor.parquet(destino)

        linhas = resultado.count()
        duracao = (datetime.now(timezone.utc) - inicio).total_seconds()
        metricas.append({"tabela": nome, "linhas": linhas, "segundos": round(duracao, 2),
                         "destino": destino, "status": "OK"})
        log.info("[GOLD] %-30s | %8d linhas | %6.2fs -> %s", nome, linhas, duracao, destino)
    except Exception as exc:
        if spec.get("opcional"):
            log.warning("[GOLD] %s falhou e é opcional: %s", nome, exc)
            metricas.append({"tabela": nome, "linhas": 0, "segundos": 0,
                             "destino": "-", "status": f"IGNORADA: {exc}"})
        else:
            log.error("[GOLD] %s falhou: %s", nome, exc)
            raise

log.info("=" * 70)
log.info("GOLD CONCLUIDA | %s", json.dumps(metricas, ensure_ascii=False))
log.info("Proxima etapa: Glue Crawler -> Athena (ver cloud/aws/athena/ddl_gold.sql)")
log.info("=" * 70)

job.commit()

In [ ]:
%%writefile cloud/aws/athena/ddl_gold.sql
-- ============================================================================
-- Athena — Camada GOLD da pipeline de alfabetização
-- ============================================================================
-- Como usar:
--   1. Rode o Glue Crawler sobre s3://<conta>-tc2-data-spec/gold/ (recomendado:
--      ele infere o schema e registra as partições automaticamente), OU
--   2. Execute os DDLs abaixo e, depois, MSCK REPAIR TABLE para carregar as partições.
--
-- FinOps: todas as consultas de exemplo filtram por `ano` (coluna de partição) e
-- projetam apenas as colunas necessárias. Cobrança do Athena = bytes escaneados.
-- ============================================================================

CREATE DATABASE IF NOT EXISTS tc2_alfabetizacao
COMMENT 'Indicador Crianca Alfabetizada - camada Gold'
LOCATION 's3://SUA-CONTA-tc2-data-spec/gold/';

-- ----------------------------------------------------------------------------
-- Tabela principal: indicador por municipio
-- ----------------------------------------------------------------------------
CREATE EXTERNAL TABLE IF NOT EXISTS tc2_alfabetizacao.gold_indicador_municipio (
    id_municipio              int,
    nome_municipio            string,
    sigla_uf                  string,
    nome_uf                   string,
    nome_regiao               string,
    nome_mesorregiao          string,
    rede                      int,
    rede_nome                 string,
    serie                     int,
    taxa_alfabetizacao        double,
    media_portugues           double,
    percentual_participacao   double,
    meta_taxa_ano             double,
    meta_taxa_2030            double,
    gap_meta_ano              double,
    gap_meta_2030             double,
    taxa_uf                   double,
    taxa_brasil_publica       double,
    dif_vs_uf                 double,
    dif_vs_brasil             double,
    risco_alfabetizacao       string,
    situacao_meta             string,
    ritmo_anual_necessario_pp double,
    posicao_na_uf             int,
    quartil_nacional          int
)
PARTITIONED BY (ano int)
STORED AS PARQUET
LOCATION 's3://SUA-CONTA-tc2-data-spec/gold/gold_indicador_municipio/'
TBLPROPERTIES ('parquet.compression' = 'SNAPPY');

MSCK REPAIR TABLE tc2_alfabetizacao.gold_indicador_municipio;

-- ----------------------------------------------------------------------------
-- Consolidado meta x realizado por UF
-- ----------------------------------------------------------------------------
CREATE EXTERNAL TABLE IF NOT EXISTS tc2_alfabetizacao.gold_meta_vs_realizado (
    rede                    int,
    rede_nome               string,
    sigla_uf                string,
    nome_regiao             string,
    municipios              bigint,
    taxa_media              double,
    meta_media              double,
    gap_medio_pp            double,
    gap_medio_2030_pp       double,
    pct_municipios_na_meta  double,
    municipios_criticos     bigint,
    participacao_media      double
)
PARTITIONED BY (ano int)
STORED AS PARQUET
LOCATION 's3://SUA-CONTA-tc2-data-spec/gold/gold_meta_vs_realizado/'
TBLPROPERTIES ('parquet.compression' = 'SNAPPY');

MSCK REPAIR TABLE tc2_alfabetizacao.gold_meta_vs_realizado;

-- ============================================================================
-- CONSULTAS DE NEGÓCIO
-- ============================================================================

-- 1. Municípios em risco crítico no ciclo mais recente (prioridade de intervenção)
SELECT sigla_uf, nome_municipio, taxa_alfabetizacao, meta_taxa_ano, gap_meta_2030
FROM tc2_alfabetizacao.gold_indicador_municipio
WHERE ano = 2024
  AND rede_nome = 'Municipal'
  AND risco_alfabetizacao = 'Critico'
ORDER BY taxa_alfabetizacao ASC
LIMIT 100;

-- 2. Ranking de estados por percentual de municípios dentro da meta
SELECT sigla_uf, nome_regiao, municipios, taxa_media, pct_municipios_na_meta, municipios_criticos
FROM tc2_alfabetizacao.gold_meta_vs_realizado
WHERE ano = 2024 AND rede_nome = 'Municipal'
ORDER BY pct_municipios_na_meta DESC;

-- 3. Esforço anual necessário até 2030, por região
SELECT nome_regiao,
       COUNT(*)                             AS municipios,
       ROUND(AVG(taxa_alfabetizacao), 2)    AS taxa_media,
       ROUND(AVG(ritmo_anual_necessario_pp), 2) AS ritmo_anual_necessario_pp
FROM tc2_alfabetizacao.gold_indicador_municipio
WHERE ano = 2024 AND rede_nome = 'Municipal'
GROUP BY nome_regiao
ORDER BY ritmo_anual_necessario_pp DESC;

-- 4. Municípios que estão acima da meta do ano mas abaixo da média da própria UF
--    (bom no absoluto, ruim no relativo — recorte útil para focalizar apoio técnico)
SELECT sigla_uf, nome_municipio, taxa_alfabetizacao, taxa_uf, dif_vs_uf
FROM tc2_alfabetizacao.gold_indicador_municipio
WHERE ano = 2024 AND rede_nome = 'Municipal'
  AND situacao_meta = 'Meta atingida'
  AND dif_vs_uf < 0
ORDER BY dif_vs_uf ASC
LIMIT 50;

-- ============================================================================
-- VIEW pronta para dashboard (QuickSight / Power BI)
-- ============================================================================
CREATE OR REPLACE VIEW tc2_alfabetizacao.vw_painel_alfabetizacao AS
SELECT ano, sigla_uf, nome_regiao, nome_municipio, rede_nome,
       taxa_alfabetizacao, meta_taxa_ano, gap_meta_ano, gap_meta_2030,
       risco_alfabetizacao, situacao_meta, posicao_na_uf
FROM tc2_alfabetizacao.gold_indicador_municipio
WHERE rede_nome IN ('Municipal', 'Publica', 'Pública');

In [ ]:
%%writefile cloud/aws/streaming/lambda_stream_processor.py
"""AWS Lambda — processador de eventos de avaliação (gatilho Amazon MSK).

É a versão gerenciada do stream processor do Notebook 02: valida contra o
contrato, roteia inválidos para a DLQ (SQS), enriquece e publica alertas no SNS.
Os eventos válidos seguem para o Firehose, que faz o buffer e escreve Parquet
particionado no S3 — sem código de micro-batch e sem small files.

Variáveis de ambiente:
    DLQ_URL              URL da fila SQS de dead letter
    SNS_TOPIC_ARN        tópico de alertas
    FIREHOSE_STREAM      nome do delivery stream para a Bronze
    TABELA_MUNICIPIOS    tabela DynamoDB com a dimensão territorial (cache)
"""
import base64
import json
import os
from datetime import datetime, timezone

import boto3

PONTO_CORTE_SAEB = 743
CAMPOS_OBRIGATORIOS = ["id_evento", "tipo_evento", "ts_evento", "id_municipio",
                       "rede", "proficiencia_portugues"]

sqs = boto3.client("sqs")
sns = boto3.client("sns")
firehose = boto3.client("firehose")
dynamodb = boto3.resource("dynamodb")

DLQ_URL          = os.environ["DLQ_URL"]
SNS_TOPIC_ARN    = os.environ["SNS_TOPIC_ARN"]
FIREHOSE_STREAM  = os.environ["FIREHOSE_STREAM"]
tabela_municipios = dynamodb.Table(os.environ["TABELA_MUNICIPIOS"])

# Cache em memória do container: a dimensão territorial muda raramente, e uma
# leitura no DynamoDB por evento seria o maior custo da função.
_cache_municipios = {}


def buscar_municipio(id_municipio):
    if id_municipio not in _cache_municipios:
        resposta = tabela_municipios.get_item(Key={"id_municipio": int(id_municipio)})
        _cache_municipios[id_municipio] = resposta.get("Item")
    return _cache_municipios[id_municipio]


def validar(evento):
    faltantes = [c for c in CAMPOS_OBRIGATORIOS if evento.get(c) is None]
    if faltantes:
        return False, "campos_obrigatorios_ausentes:" + ",".join(faltantes)
    try:
        proficiencia = float(evento["proficiencia_portugues"])
    except (TypeError, ValueError):
        return False, "proficiencia_nao_numerica"
    if not 200 <= proficiencia <= 1000:
        return False, "proficiencia_fora_de_faixa:%s" % proficiencia
    if buscar_municipio(evento["id_municipio"]) is None:
        return False, "municipio_inexistente:%s" % evento["id_municipio"]
    return True, None


def lambda_handler(event, context):
    """Gatilho MSK: `event['records']` traz lotes por partição do tópico."""
    validos, rejeitados = [], 0

    for _particao, mensagens in event.get("records", {}).items():
        for mensagem in mensagens:
            bruto = json.loads(base64.b64decode(mensagem["value"]).decode("utf-8"))
            ok, motivo = validar(bruto)

            if not ok:
                rejeitados += 1
                sqs.send_message(QueueUrl=DLQ_URL, MessageBody=json.dumps({
                    "evento_original": bruto,
                    "motivo_rejeicao": motivo,
                    "ts_rejeicao": datetime.now(timezone.utc).isoformat(),
                }))
                continue

            territorio = buscar_municipio(bruto["id_municipio"])
            enriquecido = dict(bruto)
            enriquecido.update({
                "nome_municipio": territorio["nome_municipio"],
                "sigla_uf": territorio["sigla_uf"],
                "nome_regiao": territorio["nome_regiao"],
                "alfabetizado": float(bruto["proficiencia_portugues"]) >= PONTO_CORTE_SAEB,
                "ts_processamento": datetime.now(timezone.utc).isoformat(),
            })
            validos.append({"Data": (json.dumps(enriquecido) + "\n").encode("utf-8")})

    # Firehose faz o buffer por tempo/tamanho e converte para Parquet no S3.
    for inicio in range(0, len(validos), 500):        # limite da API: 500 registros por chamada
        firehose.put_record_batch(DeliveryStreamName=FIREHOSE_STREAM,
                                  Records=validos[inicio:inicio + 500])

    total = len(validos) + rejeitados
    taxa_dlq = (rejeitados / total * 100) if total else 0
    if taxa_dlq > 10:
        sns.publish(
            TopicArn=SNS_TOPIC_ARN,
            Subject="[ALERTA] Taxa de rejeicao acima do limite",
            Message=json.dumps({"tipo": "TAXA_DLQ_ALTA", "taxa_dlq_pct": round(taxa_dlq, 2),
                                "eventos": total, "rejeitados": rejeitados}),
        )

    return {"processados": total, "validos": len(validos), "rejeitados": rejeitados}

In [ ]:
%%writefile cloud/aws/glue/orquestracao_step_functions.json
{
  "Comment": "Pipeline TC2 - Alfabetizacao: Bronze -> Silver -> Gold -> Catalogo, com alerta em falha",
  "StartAt": "IngestaoBronze",
  "States": {
    "IngestaoBronze": {
      "Type": "Task",
      "Resource": "arn:aws:states:::glue:startJobRun.sync",
      "Parameters": {
        "JobName": "tc2-etl-bronze",
        "Arguments": {
          "--BUCKET_LANDING": "SUA-CONTA-tc2-landing",
          "--BUCKET_SOR": "SUA-CONTA-tc2-data-sor",
          "--ENTIDADES": "indicador_municipio,indicador_uf,meta_brasil,meta_uf,meta_municipio",
          "--API_IBGE": "https://servicodados.ibge.gov.br/api/v1/localidades"
        }
      },
      "Retry": [
        {
          "ErrorEquals": ["Glue.ConcurrentRunsExceededException", "States.TaskFailed"],
          "IntervalSeconds": 60,
          "MaxAttempts": 2,
          "BackoffRate": 2.0
        }
      ],
      "Catch": [
        { "ErrorEquals": ["States.ALL"], "Next": "NotificarFalha", "ResultPath": "$.erro" }
      ],
      "Next": "TratamentoSilver"
    },

    "TratamentoSilver": {
      "Type": "Task",
      "Resource": "arn:aws:states:::glue:startJobRun.sync",
      "Parameters": {
        "JobName": "tc2-etl-silver",
        "Arguments": {
          "--BUCKET_SOR": "SUA-CONTA-tc2-data-sor",
          "--BUCKET_SOT": "SUA-CONTA-tc2-data-sot"
        }
      },
      "Catch": [
        { "ErrorEquals": ["States.ALL"], "Next": "NotificarFalha", "ResultPath": "$.erro" }
      ],
      "Next": "CamadaGold"
    },

    "CamadaGold": {
      "Type": "Task",
      "Resource": "arn:aws:states:::glue:startJobRun.sync",
      "Parameters": {
        "JobName": "tc2-etl-gold",
        "Arguments": {
          "--BUCKET_SOT": "SUA-CONTA-tc2-data-sot",
          "--BUCKET_SPEC": "SUA-CONTA-tc2-data-spec",
          "--TABELAS": "todas"
        }
      },
      "Catch": [
        { "ErrorEquals": ["States.ALL"], "Next": "NotificarFalha", "ResultPath": "$.erro" }
      ],
      "Next": "AtualizarCatalogo"
    },

    "AtualizarCatalogo": {
      "Type": "Task",
      "Resource": "arn:aws:states:::aws-sdk:glue:startCrawler",
      "Parameters": { "Name": "tc2-crawler-gold" },
      "Catch": [
        { "ErrorEquals": ["States.ALL"], "Next": "NotificarFalha", "ResultPath": "$.erro" }
      ],
      "Next": "NotificarSucesso"
    },

    "NotificarSucesso": {
      "Type": "Task",
      "Resource": "arn:aws:states:::sns:publish",
      "Parameters": {
        "TopicArn": "arn:aws:sns:us-east-1:SUA-CONTA:tc2-pipeline-status",
        "Subject": "[OK] Pipeline TC2 Alfabetizacao concluida",
        "Message.$": "States.Format('Execucao {} concluida com sucesso.', $$.Execution.Name)"
      },
      "End": true
    },

    "NotificarFalha": {
      "Type": "Task",
      "Resource": "arn:aws:states:::sns:publish",
      "Parameters": {
        "TopicArn": "arn:aws:sns:us-east-1:SUA-CONTA:tc2-pipeline-alertas",
        "Subject": "[FALHA] Pipeline TC2 Alfabetizacao",
        "Message.$": "States.Format('Execucao {} falhou: {}', $$.Execution.Name, $.erro.Cause)"
      },
      "Next": "FalhaExecucao"
    },

    "FalhaExecucao": {
      "Type": "Fail",
      "Error": "PipelineTC2Falhou",
      "Cause": "Uma das etapas do pipeline falhou. Ver detalhes no SNS e no CloudWatch Logs."
    }
  }
}

In [ ]:
# ============================================================
# ETAPA 7 — QUALITY GATE DA CAMADA GOLD E PAINEL FINAL
# ============================================================
# A Gold também passa por gate: é o que chega ao gestor e ao modelo.
from src.quality.data_quality import DataQualityCheck, consolidar_relatorios

relatorios_gold = []

with monitor.etapa("quality_gate:gold", camada="gold") as metricas:
    indicador = ler_parquet(GOLD / "gold_indicador_municipio")
    dq_indicador = (DataQualityCheck(indicador, "gold_indicador_municipio", "gold",
                                     output_dir=QUALITY)
                    .check_row_count(10_000)
                    .check_chave_composta(["ano", "id_municipio", "rede"])
                    .check_not_null(["ano", "id_municipio", "nome_municipio", "sigla_uf",
                                     "taxa_alfabetizacao", "risco_alfabetizacao", "situacao_meta"])
                    .check_domain("risco_alfabetizacao", ["Critico", "Alto", "Medio", "Baixo"])
                    .check_domain("situacao_meta",
                                  ["Meta atingida", "Abaixo da meta", "Sem meta definida"])
                    .check_range("taxa_alfabetizacao", 0, 100)
                    .check_range("quartil_nacional", 1, 4))
    relatorios_gold.append(dq_indicador.relatorio())
    dq_indicador.aprovar_ou_falhar()

    consolidado = ler_parquet(GOLD / "gold_meta_vs_realizado")
    dq_consolidado = (DataQualityCheck(consolidado, "gold_meta_vs_realizado", "gold",
                                       output_dir=QUALITY)
                      .check_chave_composta(["ano", "rede", "sigla_uf"])
                      .check_not_null(["sigla_uf", "municipios", "taxa_media"])
                      .check_range("taxa_media", 0, 100)
                      .check_range("pct_municipios_na_meta", 0, 100))
    relatorios_gold.append(dq_consolidado.relatorio(imprimir=False))
    dq_consolidado.aprovar_ou_falhar()

    features = ler_parquet(GOLD / "gold_features_ml")
    dq_features = (DataQualityCheck(features, "gold_features_ml", "gold", output_dir=QUALITY)
                   .check_unique(["id_municipio"])
                   .check_not_null(["id_municipio", "alvo_taxa_alfabetizacao"])
                   .check_range("alvo_taxa_alfabetizacao", 0, 100)
                   .check_domain("alvo_atingiu_meta", [0, 1])
                   # Features com nulo não impedem o treino (imputação no Notebook 04),
                   # mas precisam ser visíveis para quem for modelar.
                   .check_not_null(["taxa_ciclo_anterior"], severidade="WARNING"))
    relatorios_gold.append(dq_features.relatorio(imprimir=False))
    dq_features.aprovar_ou_falhar()

    # ── Variação entre os ciclos ──
    # Uma UF inteira andando na contramão do país normalmente indica mudança de
    # metodologia na origem, não queda real. O Notebook 04 trata o caso.
    evolucao = ler_parquet(GOLD / "gold_evolucao_temporal")
    evolucao_municipal = evolucao[(evolucao["rede_nome"] == "Municipal")
                                  & evolucao["variacao_pp"].notna()]
    mediana_nacional = evolucao_municipal["variacao_pp"].median()
    por_uf = evolucao_municipal.groupby("sigla_uf")["variacao_pp"].mean()
    ufs_anomalas = por_uf[(por_uf - mediana_nacional).abs() > 15]

    dq_evolucao = (DataQualityCheck(evolucao_municipal, "gold_evolucao_temporal", "gold",
                                    output_dir=QUALITY)
                   .check_not_null(["id_municipio", "taxa_final"])
                   .check_domain("tendencia", ["Melhora", "Piora", "Estavel", "Serie unica"])
                   .check_expectativa(
                       "consistencia_entre_ciclos:uf",
                       len(ufs_anomalas) == 0,
                       f"{len(ufs_anomalas)} UF(s) com variação média incompatível com a mediana "
                       f"nacional ({mediana_nacional:+.2f} p.p.): "
                       + (", ".join(f"{uf} ({valor:+.1f})" for uf, valor in ufs_anomalas.items())
                          or "nenhuma"),
                       dimensao="CONSISTENCIA", severidade="WARNING"))
    relatorios_gold.append(dq_evolucao.relatorio(imprimir=bool(len(ufs_anomalas))))
    dq_evolucao.aprovar_ou_falhar()

    if len(ufs_anomalas):
        monitor.registrar_alerta(
            "WARNING",
            f"Possível quebra de série histórica em: {', '.join(ufs_anomalas.index)}. "
            "Tratada explicitamente na Etapa 3.3 do Notebook 04.")

    metricas["linhas_entrada"] = len(indicador) + len(consolidado) + len(features)

monitor.imprimir_painel()
monitor.salvar()

print("\nPAINEL DE QUALIDADE — CAMADA GOLD")
display(consolidar_relatorios(relatorios_gold))

---
## Artefatos gerados para deploy

```
cloud/aws/glue/etl_bronze.py                    job Glue de ingestão
cloud/aws/glue/etl_silver.py                    job Glue de padronização e integração
cloud/aws/glue/etl_gold.py                      job Glue que executa src/gold/consultas.py
cloud/aws/glue/orquestracao_step_functions.json máquina de estados com retry e alerta
cloud/aws/athena/ddl_gold.sql                   DDLs, consultas de negócio e view de painel
cloud/aws/streaming/lambda_stream_processor.py  processador MSK para DLQ, Firehose e SNS

src/gold/consultas.py                           regra de negócio da Gold
data/lake/gold/                                 8 tabelas em Parquet particionado
```

O notebook 04 consome a Gold para analisar a desigualdade educacional e treinar os modelos.